# 05 · Climate_TemperaturaMinima · consolidación estación-día

Consolida **Temperatura mínima** después de la auditoría diaria. No imputa valores y exige cobertura evaluable.

In [ ]:
from pathlib import Path
import subprocess
import sys

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = 'https://github.com/cybercolombia/suelosabio.git'
REPO_REF = 'feature/SCRUM-16'
REPO_DIR = Path('/content/suelosabio') if IN_COLAB else Path.cwd()
if IN_COLAB:
    if not (REPO_DIR / '.git').exists():
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
    else:
        remote_ref = f'refs/remotes/origin/{REPO_REF}'
        subprocess.run(['git', 'fetch', '--depth', '1', 'origin', f'+refs/heads/{REPO_REF}:{remote_ref}'], cwd=REPO_DIR, check=True)
        subprocess.run(['git', 'checkout', '-B', REPO_REF, remote_ref], cwd=REPO_DIR, check=True)
    PIPELINE_DIR = REPO_DIR / 'notebooks' / 'ClimatePipeline'
else:
    candidatos = [REPO_DIR / 'notebooks' / 'ClimatePipeline', REPO_DIR / 'ClimatePipeline', REPO_DIR]
    PIPELINE_DIR = next((p for p in candidatos if (p / 'DatasetConfig.py').exists()), None)
if PIPELINE_DIR is None:
    raise FileNotFoundError('No se encontró notebooks/ClimatePipeline.')
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))


In [ ]:
import json
import time
import pandas as pd
from ClimateProcessingUtils import ahora_proyecto, detectar_commit, escribir_json_atomico, escribir_parquet_atomico, slugificar
from DatasetConfig import cargar_configuracion_datasets
from ScalarDailyConsolidation import CONSOLIDATION_VERSION, consolidar_escalar_diario

VARIABLE_NOMBRE = 'temperatura_minima'
DATASET_ID = 'afdg-3zpb'
AUDITORIA_NOMBRE = 'cierre_temperatura_minima_2024_2025_v1'
CONSOLIDACION_NOMBRE = 'cierre_temperatura_minima_2024_2025_v1'
COBERTURA_MINIMA_PCT = 90.0
COBERTURA_MAXIMA_PCT = 102.0
EJECUTAR_CONSOLIDACION = False
GUARDAR_RESULTADOS = True
SOBRESCRIBIR_RESULTADOS = False

CONFIG = cargar_configuracion_datasets(in_colab=IN_COLAB)
AUDIT_INPUT_DIR = CONFIG.processed_root / 'auditorias_clima_diario' / f'variable={VARIABLE_NOMBRE}' / f'fuente={DATASET_ID}' / f'auditoria={slugificar(AUDITORIA_NOMBRE)}'
OUTPUT_DIR = CONFIG.processed_root / 'clima_diario_curado' / f'variable={VARIABLE_NOMBRE}' / f'fuente={DATASET_ID}' / f'consolidacion={slugificar(CONSOLIDACION_NOMBRE)}'
print({'variable': VARIABLE_NOMBRE, 'entrada': str(AUDIT_INPUT_DIR), 'salida': str(OUTPUT_DIR), 'ejecutar': EJECUTAR_CONSOLIDACION})


In [ ]:
if not EJECUTAR_CONSOLIDACION:
    print('Consolidación desactivada. Cambie EJECUTAR_CONSOLIDACION a True después de revisar el plan.')
else:
    inicio_reloj = time.perf_counter()
    inicio = ahora_proyecto()
    manifest_path = AUDIT_INPUT_DIR / 'manifest.json'
    if not manifest_path.exists():
        raise FileNotFoundError(f'No existe {manifest_path}')
    manifest_entrada = json.loads(manifest_path.read_text(encoding='utf-8'))
    if manifest_entrada.get('estado') != 'COMPLETA':
        raise RuntimeError('La auditoría diaria de entrada no está completa.')
    calendario = pd.read_parquet(AUDIT_INPUT_DIR / 'calendario_estacion_sensor.parquet')
    sospechosos_path = AUDIT_INPUT_DIR / 'valores_sospechosos.parquet'
    sospechosos = pd.read_parquet(sospechosos_path) if sospechosos_path.exists() else pd.DataFrame()
    resultado = consolidar_escalar_diario(
        calendario,
        sospechosos,
        cobertura_minima_pct=COBERTURA_MINIMA_PCT,
        cobertura_maxima_pct=COBERTURA_MAXIMA_PCT,
    )
    if GUARDAR_RESULTADOS:
        salida_manifest = OUTPUT_DIR / 'manifest.json'
        if salida_manifest.exists() and not SOBRESCRIBIR_RESULTADOS:
            previo = json.loads(salida_manifest.read_text(encoding='utf-8'))
            if previo.get('estado') == 'COMPLETA':
                raise FileExistsError(f'La salida ya está completa: {OUTPUT_DIR}')
        diario = resultado.diario_estacion.assign(anio=lambda x: x.fecha.dt.year, mes=lambda x: x.fecha.dt.month)
        particiones=[]
        for (departamento, anio, mes), bloque in diario.groupby(['departamento','anio','mes'], sort=True):
            ruta = OUTPUT_DIR / f'departamento={departamento}' / f'anio={int(anio)}' / f'mes={int(mes):02d}' / 'observaciones_estacion_dia.parquet'
            escribir_parquet_atomico(bloque.drop(columns=['anio','mes']), ruta, sobrescribir=SOBRESCRIBIR_RESULTADOS)
            particiones.append({'ruta':str(ruta),'filas':len(bloque)})
        escribir_parquet_atomico(resultado.candidatos_sensor, OUTPUT_DIR / 'candidatos_sensor.parquet', sobrescribir=SOBRESCRIBIR_RESULTADOS)
        fin = ahora_proyecto()
        escribir_json_atomico({'estado':'COMPLETA','regla_version':CONSOLIDATION_VERSION,'variable':VARIABLE_NOMBRE,'dataset_id':DATASET_ID,'commit':detectar_commit(REPO_DIR),'inicio':inicio.isoformat(),'fin':fin.isoformat(),'duracion_segundos':round(time.perf_counter()-inicio_reloj,2),'metricas':resultado.metricas,'particiones':particiones,'entrada':str(AUDIT_INPUT_DIR)}, salida_manifest, sobrescribir=True)
        print(f'Consolidación guardada en {OUTPUT_DIR}')
    display(resultado.diario_estacion.head())
